# 082 — Cuantización e inferencia local

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** s = 0,50/127 ≈ 0,003937. q = round(w/s) = [30, −127, 64, −25].
x̂ = [0,1181, −0,5000, 0,2520, −0,0984]. Errores ≈ [0,0019, 0, 0,0020, 0,0016];
máximo 0,0020 ≤ s/2 = 0,00197 ✓ (dentro del margen de redondeo).

**Ejercicio 2.** Global: s = 8,0/127 ≈ 0,063 → q = [2, −8, 4, −2, 127],
x̂ = [0,126, −0,504, 0,252, −0,126]; error máximo de los pequeños ≈ **0,026**
(~13× peor). Por grupos: el grupo pequeño conserva s = 0,003937 (error ≤ 0,002) y
el outlier tiene su propia escala exacta. Conclusión: las escalas por bloque
aíslan el daño — el mecanismo central de Q4_K, GPTQ y NF4.

**Ejercicio 3.** FP16: 16 GB; INT8: 8 GB; Q4_K_M: 8e9·4,55/8 ≈ **4,6 GB**.
Solo el Q4 cabe en el presupuesto de ~6 GB útiles del portátil.

**Ejercicio 4.** Ejemplo de limitación honesta: "la calidad se evaluó con
perplejidad agregada, no con casos del dominio; la cuantización puede degradar
justo las respuestas largas y técnicas que este dominio exige; se requiere
evaluación específica y revisión humana".

In [ ]:
# Ejercicios 1 y 2
def quant(ws):
    s = max(abs(x) for x in ws) / 127
    q = [max(-127, min(127, round(x / s))) for x in ws]
    xh = [s * qi for qi in q]
    return s, q, xh

w = [0.12, -0.50, 0.25, -0.10]
s, q, xh = quant(w)
print("s=", round(s, 6), "q=", q, "err_max=",
      max(abs(a - b) for a, b in zip(w, xh)))

s2, q2, xh2 = quant(w + [8.0])
print("con outlier, err pequeños:",
      max(abs(a - b) for a, b in zip(w, xh2[:4])))  # ~0.026, 13x peor

# Ejercicio 3
N = 8e9
for nombre, bits in [("FP16", 16), ("INT8", 8), ("Q4_K_M", 4.55)]:
    print(nombre, round(N * bits / 8 / 1e9, 1), "GB")

# Ejercicio 4
result = run_lab("neural", seed=82)
assert result["kind"] == "neural"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué la cuantización acelera el decode aunque la GPU tenga que descuantizar
   cada peso antes de multiplicar?
2. Un outlier de magnitud 12 entre pesos de magnitud <1 multiplica el error de
   todos: ¿qué dos técnicas distintas de la clase lo neutralizan y en qué difieren?
3. ¿En qué escenarios el modelo local en Q4 supera *en la práctica* a un modelo
   mayor vía API, aun perdiendo en benchmarks de calidad?